<a href="https://colab.research.google.com/github/Muhammad-Ahmad-1263/urdu-ocr-codesaviours-si26-Muhammad-Ahmad/blob/main/SI26_Week5_Muhammad_Ahmad.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Urdu OCR — Week 5
### Code Saviours (SMC-PRIVATE) Limited — ML / AI Internship, Batch SI-26

**Author:** Muhammad Ahmad

**Goal:** Wrap the Week 4 fine-tuned TrOCR model in a Gradio web interface, test it live in Colab, then deploy it permanently on HuggingFace Spaces and document it on GitHub.

**This notebook = Step 1 only.** Steps 2 (HuggingFace Spaces) and 3 (README) happen outside Colab — see the final section for a recap and links.

| Step | Where | Status after this notebook |
|---|---|---|
| 1. Build & test Gradio app | This notebook | Done once you see extracted text below |
| 2. Deploy on HuggingFace Spaces | huggingface.co | To do after this notebook |
| 3. Write README | GitHub | To do after Step 2 |

## 1. Install dependencies
Run this once per session. If you restart the Colab runtime, run it again.

In [1]:
!pip uninstall -y torchaudio

In [2]:
!pip install -q -U torch torchvision transformers tokenizers huggingface_hub gradio pillow sentencepiece
print('Dependencies installed.')
print('If this is your first install in this runtime, restart the session now (Runtime -> Restart session) before continuing to the next cell.')

Dependencies installed.
If this is your first install in this runtime, restart the session now (Runtime -> Restart session) before continuing to the next cell.


## 2. Mount Google Drive
This gives Colab access to the model you fine-tuned and saved in Week 4.

In [3]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted.


## 3. Load the fine-tuned model

This model uses a `RobertaTokenizer` (vocab.json + merges.txt), not a sentencepiece tokenizer. The tokenizer and image processor are loaded explicitly by class below instead of letting `TrOCRProcessor.from_pretrained` auto-detect them — this avoids a known bug in newer `transformers` versions that incorrectly tries to convert BPE tokenizers through a sentencepiece path.

If this cell errors with something like *"does not appear to have a file named config.json"*, your model isn't at the path below — go check Google Drive, find the real folder, and update `MODEL_PATH` to match exactly before re-running.

In [4]:
import os
from transformers import RobertaTokenizer, ViTImageProcessor, TrOCRProcessor, VisionEncoderDecoderModel
import torch

MODEL_PATH = '/content/drive/MyDrive/SI26-urdu-ocr-model'  # <-- update this if your folder name/location is different

if not os.path.isdir(MODEL_PATH):
    raise FileNotFoundError(
        f"Couldn't find a folder at {MODEL_PATH}.\n"
        "Open Google Drive, locate your Week 4 model folder, and update MODEL_PATH above to match it exactly."
    )

print('Loading tokenizer, image processor, and model — this can take a minute...')

tokenizer = RobertaTokenizer.from_pretrained(MODEL_PATH)
image_processor = ViTImageProcessor.from_pretrained(MODEL_PATH)
processor = TrOCRProcessor(image_processor=image_processor, tokenizer=tokenizer)
model = VisionEncoderDecoderModel.from_pretrained(MODEL_PATH)
model.eval()

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)
print(f'Model loaded on {device}.')

Loading tokenizer, image processor, and model — this can take a minute...


Loading weights:   0%|          | 0/480 [00:01<?, ?it/s]

Model loaded on cuda.


## 4. Define the OCR function
This is the one function Gradio needs — give it an image, get back Urdu text.

In [5]:
def extract_urdu_text(image):
    """Takes a PIL image, returns extracted Urdu text (or a friendly message)."""
    if image is None:
        return 'Please upload an image.'
    try:
        pixel_values = processor(image, return_tensors='pt').pixel_values.to(device)
        with torch.no_grad():
            generated_ids = model.generate(pixel_values)
        text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
        return text if text.strip() else 'Could not extract any text from this image.'
    except Exception as e:
        return f'Something went wrong while processing this image: {e}'

## 5. (Optional) Quick sanity check before launching the full app
Uploads happen through the Gradio UI below, but if you want to test the function directly on one image first, run this cell and follow the upload prompt.

In [7]:
from google.colab import files
from PIL import Image
import io

print('Upload one test image to sanity-check the model (optional — you can skip this cell).')
uploaded = files.upload()
for fname, fdata in uploaded.items():
    img = Image.open(io.BytesIO(fdata)).convert('RGB')
    print(f'\n--- {fname} ---')
    print('Extracted text:', extract_urdu_text(img))

Upload one test image to sanity-check the model (optional — you can skip this cell).


Saving Screenshot 2026-07-13 113449.png to Screenshot 2026-07-13 113449.png

--- Screenshot 2026-07-13 113449.png ---


/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1638: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


Extracted text: ببمدگٹ ابببببککے تت


## 6. Build and launch the Gradio interface
This prints a public link ending in `.gradio.live`. It stays live only while this notebook is running.

In [8]:
import gradio as gr

interface = gr.Interface(
    fn=extract_urdu_text,
    inputs=gr.Image(type='pil', label='Upload Urdu Image'),
    outputs=gr.Textbox(label='Extracted Urdu Text'),
    title='Urdu OCR -- Code Saviours SI-26',
    description='Upload an image containing Urdu text and get the extracted text.',
    examples=[],
)

interface.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e9da916fa87d400573.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


**Checklist before moving on:**
- [ ] Clicked the printed `.gradio.live` link
- [ ] Uploaded a real Urdu test image
- [ ] Got Urdu text back correctly
- [ ] Took a screenshot of the working demo (you'll need it for your README/report)

---
## Step 2 — Deploy on HuggingFace Spaces *(done outside this notebook)*

1. huggingface.co → profile icon → **New Space**
2. Name: `urdu-ocr-codesaviours-si26-muhammad` · SDK: **Gradio** · Visibility: **Public**
3. Add `app.py` — same logic as above, but **no Drive mount** (model files are uploaded directly to the Space) and use `interface.launch()` without `share=True`
4. Add `requirements.txt`:
   ```
   transformers==4.35.0
   torch==2.0.1
   gradio==3.50.0
   Pillow==10.0.0
   ```
5. Upload your model files (config.json, weights, tokenizer files) to the Space root
6. Wait for the build to finish, confirm status = **Running**

*(Ready-to-use `app.py` and `requirements.txt` were shared separately in chat.)*

---
## Step 3 — Write the README *(done on GitHub)*

Sections, in order: title & one-liner → problem & why it matters → how it works → live demo link → run locally → dataset details → results → credit.

*(Full README draft was shared separately in chat — fill in dataset size/accuracy and commit.)*